# Q6 Destatis: Population Germany (12411-0001) – Clean

**Goal of Clean:** turn the raw Destatis file into a small table (year, population, 2001–2025) ready to be analysed. We change nothing in `data/raw/`.

Source: © Statistisches Bundesamt (Destatis), 2026, License dl-de/by-2-0.

In [1]:
import pandas as pd

RAW = "../../data/raw/destatis/12411-0001_de.csv"                     # raw file (we only read it, never change it)
OUT = "../../data/processed/q6_destatis_population.csv"               # the cleaned file we create

# 1) read, like in Explore
# The raw file is not a plain table: it has a header and other title lines before the data and footnotes at the end.
# First, the file has to be read as plain text, line after line.
with open(RAW, encoding="utf-8-sig") as f:                            # utf-8-sig=UTF-8, removes invisible BOM character at the start, if there is one
    lines = f.read().splitlines()                                     # list with one entry per line of the file

# data lines start with a date, meaning two digits
# header, title lines and footnotes start with letters or other non-digit characters, so they are not matched
# only works because no title line or footnote starts with two digits
first_data = next(i for i, l in enumerate(lines) if l[:2].isdigit())  # number of the first data line
last_data = max(i for i, l in enumerate(lines) if l[:2].isdigit())    # number of the last data line

# Now pandas reads only the data lines:
# sep=";" -> columns are separated by semicolons
# header=None -> no usable header row, no line is used as column name
# skiprows=first_data -> skips everything before the first data line (title, description, unit)
# nrows=... -> reads as many rows as there are data lines (footnotes stay out)
# names=[...] -> gives the three columns specific names
df = pd.read_csv(RAW, sep=";", encoding="utf-8-sig", header=None,
                 skiprows=first_data, nrows=last_data - first_data + 1,
                 names=["date_raw", "population_raw", "value_status"])
df.head()                                                             # shows the first 5 rows to see that the table looks as expected

,date_raw,population_raw,value_status
0,31.12.1950,50958125,e
1,31.12.1951,51434777,e
2,31.12.1952,51863761,e
3,31.12.1953,52453806,e
4,31.12.1954,52943295,e


**Notes:**
- The data is read with the right encoding (`utf-8-sig`) and separator.
- The column names are set. `value_status` is still there.
- The years have not been filtered yet.

In [2]:
# 2) transform
# "31.12.1950" is only text, so it must be converted into a real date (day.month.year) and only the year is kept.
# The year is the key to join with other tables later.
df["year"] = pd.to_datetime(df["date_raw"], format="%d.%m.%Y").dt.year
clean = (df[df["year"].between(2001, 2025)]                # keeps only rows from 2001-2025, including both years
         .rename(columns={"population_raw": "population"}) # gives population column final name
         [["year", "population"]]                          # keeps only two columns (year, population), drops population_raw and value_status
         .reset_index(drop=True))                          # numbers the rows again (after filtering they would start at 51)
print(clean.shape)                                         # (rows, columns) -> (25, 2) is expected
clean.head()

(25, 2)


,year,population
0,2001,82440309
1,2002,82536680
2,2003,82531671
3,2004,82500849
4,2005,82437995


**Notes:**
- The column `population_raw` was renamed to `population`.
- The column `year` was extracted from the column `date_raw`.
- Rows are filtered to 2001-2025 = 25 rows.
- `date_raw` and `value_status` are dropped since `year` and `population` are all we need.

In [3]:
# 3) check
# assert stops with an error if statement is False, assert stays silent if statement is True
# these lines prove that the result is what we expect
assert len(clean) == 25                                                    # 2001 to 2025 = 25 years = 25 rows
assert clean["year"].is_unique                                             # no year appears twice
assert clean.isna().sum().sum() == 0                                       # no missing values: isna() marks empty cells
                                                                           # the first sum() counts them per column
                                                                           # the second sum() adds the columns up
assert clean.loc[clean["year"] == 2025, "population"].item() == 83467117   # check against the raw file: .item() takes the single value
                                                                           # out of the one-row result
print("All checks passed")                                                 # only reached if all asserts above passed

All checks passed


**Notes:**
- Checks that there are 25 entries, that all years are unique, that we have no missing values and that the value of the last row (2025: 83,467,117) matches the raw file.

In [4]:
# 4) save
# The clean table must be written to a new CSV file.
# index=False means: do not write the row numbers as an extra column
# if the file already exists, it is overwritten, so running this cell multiple times is safe
clean.to_csv(OUT, index=False)
print("Saved to:", OUT)
pd.read_csv(OUT).head()

Saved to: ../../data/processed/q6_destatis_population.csv


,year,population
0,2001,82440309
1,2002,82536680
2,2003,82531671
3,2004,82500849
4,2005,82437995


**Notes:**
- The table is saved as a CSV file. Every run overwrites it.
- The saved file was read again as a check.
- The row numbers (index) are not saved.

**Conclusion:**
- Break in 2011: Values from 2011 onwards are based on the 2011 census, values from 2022 onwards are based on the 2022 census. Values have not been changed. Comparisons across 2011 should be made with care.
- Steps taken:
    - read --> skip header and footnotes
    - transform --> `year` calculated from `date_raw`, filter 2001-2025, `population_raw` renamed to `population`
    - check --> 25 rows, unique years, no missing values, 2025 matches the raw file
    - save --> CSV file
- Result: 25 rows, columns `year` and `population`, CSV file: q6_destatis_population.csv
- Can be joined with ticket sales through `year`.